In [43]:
import librosa
import numpy as np
from random import sample
import matplotlib.pyplot as plt
import cv2 as cv
import os
import glob
import re

In [44]:
SAMPLE_RATE = 22050
DURATION = 2
COL_COUNT = 5160
FREQ = 84
DIR = '/Users/vmenon/repos/music_encoder/data/raw'

def clean_data(raw_dir, cleaned_dir=None, overwrite=False):
    for path in [f"{raw_dir}/{path}" for path in os.listdir(raw_dir)]:
        y, sr = librosa.load(path, sr=SAMPLE_RATE, duration=DURATION)
        cqt = librosa.cqt(y, sr=sr, n_bins=FREQ)[:, :COL_COUNT]
        db_cqt = librosa.amplitude_to_db(np.abs(cqt), ref=np.max)
        cqt_max = abs(db_cqt.min())
        modified_cqt = (db_cqt + cqt_max) / cqt_max
        if modified_cqt.shape[1] < COL_COUNT:
            print(path, "is too small, skipping this song")
            diff = COL_COUNT - modified_cqt.shape[1]
            modified_cqt = np.append(modified_cqt, modified_cqt[:, 0:diff], axis=1)

        fig, ax = plt.subplots()
        print(db_cqt.shape)
        img = librosa.display.specshow(db_cqt,
                                       sr=SAMPLE_RATE, x_axis='time', y_axis='cqt_note', ax=ax)
        ax.set_title('Constant-Q power spectrum')
        fig.colorbar(img, ax=ax, format="%+2.0f dB")

def mp3s_in_dir(raw_dir):
    paths = []
    for path in glob.glob(os.path.join(raw_dir, "*.mp3")):
        paths.append(path)
    return paths

def cqts_from_paths(paths: list[str]):
    cqts = []
    for path in paths:
        y, sr = librosa.load(path, sr=SAMPLE_RATE, duration=None)
        cqt = librosa.cqt(y, sr=sr, n_bins=FREQ)
        
        

In [45]:
mp3s_in_dir(DIR)

Elton_John_Kiki_Dee__Dont_Go_Breaking_My_Heart
The_Offspring__Self_Esteem
KSI_SX_Lil_Baby_Rick_Ross__Down_Like_That_feat_Rick_Ross_Lil_Baby__SX
Chance_the_Rapper_AbSoul__Smoke_Again
JKwon__Tipsy__Club_Mix
Katy_Perry__Teenage_Dream
Steve_Lacy__C_U_Girl
Plain_White_Ts__Our_Time_Now
Chance_the_Rapper_Jeremih_Francis_and_the_Lights__Summer_Friends_feat_Jeremih__Francis__The_Lights
Rob_Thomas__Little_Wonders
The_AllAmerican_Rejects__Move_Along
Nirvana__Smells_Like_Teen_Spirit
The_Jackson_5__ABC
JAYZ_Kanye_West__Nias_In_Paris
The_Buggles__Video_Killed_The_Radio_Star
The_Notorious_BIG__Hypnotize__2014_Remaster
Katy_Perry__ET
Mac_Miller__Kool_Aid__Frozen_Pizza
Lucas_Grabeel_Sharpay_Evans_Disney__What_Ive_Been_Looking_For
Chance_the_Rapper_Justin_Bieber_Towkio__Juke_Jam_feat_Justin_Bieber__Towkio
J_Cole__MIDDLE_CHILD
Green_Day__Holiday__Boulevard_of_Broken_Dreams
J_Boog__Lets_Do_It_Again
Seal__Kiss_from_a_Rose
Wallows__Drunk_on_Halloween
Auralnauts__Dance_Fight_66
Bobby_Shmurda__Hot_Ngga
Chance

In [27]:
corners_on_image('cqt_2.png', 'cqt_3')

In [ ]:
def corners_on_image(name, dest):
    file = f"/Users/vmenon/repos/music_encoder/data/images/{name}"
    img = cv.imread(file)
    gray = cv.cvtColor(img,cv.COLOR_BGR2GRAY)
     
    # find Harris corners
    gray = np.float32(gray)
    dst = cv.cornerHarris(gray,2,3,0.04)
    dst = cv.dilate(dst,None)
    ret, dst = cv.threshold(dst,0.01*dst.max(),255,0)
    dst = np.uint8(dst)
     
    # find centroids
    ret, labels, stats, centroids = cv.connectedComponentsWithStats(dst)
     
    # define the criteria to stop and refine the corners
    criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 100, 0.001)
    corners = cv.cornerSubPix(gray,np.float32(centroids),(5,5),(-1,-1),criteria)
     
    # Now draw them
    res = np.hstack((centroids,corners))
    res = np.int64(res)
    img[res[:,1],res[:,0]]=[0,0,255]
    img[res[:,3],res[:,2]] = [0,255,0]
     
    cv.imwrite(f'{dest}_cornered.png',img)

def clean_data_mfcc(raw_dir):
    for path in [f"{raw_dir}/{path}" for path in os.listdir(raw_dir)]:
        y, sr = librosa.load(path, sr=SAMPLE_RATE, duration=DURATION)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        print("Shape of MFCCs:", mfccs.shape) # (n_mfcc, number_of_frames)

        plt.figure(figsize=(10, 4))
        librosa.display.specshow(mfccs, sr=sr, x_axis='time')
        plt.colorbar(format='%+2.0f dB')
        plt.title('MFCCs')
        plt.tight_layout()
        plt.show()
        break